In [0]:
from pyspark.sql.functions import (col, avg, sum, round, when, lit,
    month, year, date_format)
from pyspark.sql.window import Window

# ── Step 1: Read Gold fact table ───────────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.daily_inventory_fact")

# ── Step 2: Define rolling windows ────────────────────────────────────
w7  = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-7,  0))
w30 = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-30, 0))
w90 = (Window.partitionBy("store_id", "product_id")
             .orderBy("inventory_date").rowsBetween(-90, 0))

# ── Step 3: Rolling averages ───────────────────────────────────────────
demand = (fact
    .withColumn("avg_units_7d",   round(avg("units_sold").over(w7),  2))
    .withColumn("avg_units_30d",  round(avg("units_sold").over(w30), 2))
    .withColumn("avg_units_90d",  round(avg("units_sold").over(w90), 2))
    .withColumn("avg_rev_7d",     round(avg("revenue").over(w7),    2))
    .withColumn("avg_rev_30d",    round(avg("revenue").over(w30),   2))

    # KPI: Seasonality Index = 7-day avg / 90-day avg
    # > 1.0 means current week is selling faster than the 90-day trend
    .withColumn("seasonality_index",
        round(when(col("avg_units_90d") > 0,
                   col("avg_units_7d") / col("avg_units_90d"))
              .otherwise(lit(1.0)), 3))

    # KPI: Trend Direction
    # Rising: selling 10%+ faster than last month
    # Falling: selling 10%+ slower than last month
    .withColumn("trend_direction",
        when(col("avg_units_7d") > col("avg_units_30d") * 1.1,  "Rising")
       .when(col("avg_units_7d") < col("avg_units_30d") * 0.9,  "Falling")
       .otherwise("Stable"))

    # Month-level seasonality (from your data: Dec & May are peak months)
    .withColumn("month_num",  month("inventory_date"))
    .withColumn("year_num",   year("inventory_date"))
    .withColumn("year_month", date_format("inventory_date", "yyyy-MM"))

    .select(
        "store_id", "product_id", "inventory_date",
        "year_num", "month_num", "year_month",
        "units_sold",                       # daily actual
        "avg_units_7d", "avg_units_30d", "avg_units_90d",
        "avg_rev_7d",   "avg_rev_30d",
        "seasonality_index", "trend_direction"
    )
)

# ── Step 4: Write to Gold ──────────────────────────────────────────────
(demand.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year_month")
    .saveAsTable("mini_project_team_a3t7.gold.demand_intelligence")
)


In [0]:
# build_demand_intelligence.py

from pyspark.sql.functions import (
    col, avg, sum, round, when, lit,
    month, year, date_format
)
from pyspark.sql.window import Window

# ── Step 1: Read Gold fact table ──────────────────────────────────────
fact = spark.table("mini_project_team_a3t7.gold.fact_inventory_full")

# ── Step 2: Define rolling windows ────────────────────────────────────
# Partitioned by store+product so each product's trend is independent
w7  = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date").rowsBetween(-7,  0))
w30 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date").rowsBetween(-30, 0))
w90 = (Window.partitionBy("store_id", "product_id")
             .orderBy("snapshot_date").rowsBetween(-90, 0))

# ── Step 3: Rolling averages + KPIs ───────────────────────────────────
demand = (fact
    # ── Rolling demand averages ────────────────────────────────────────
    .withColumn("avg_units_7d",  round(avg("units_sold").over(w7),  2))
    .withColumn("avg_units_30d", round(avg("units_sold").over(w30), 2))
    .withColumn("avg_units_90d", round(avg("units_sold").over(w90), 2))

    # ── Rolling revenue averages ───────────────────────────────────────
    # Column is total_revenue in fact_inventory_full (was "revenue" before)
    .withColumn("avg_rev_7d",    round(avg("total_revenue").over(w7),  2))
    .withColumn("avg_rev_30d",   round(avg("total_revenue").over(w30), 2))

    # ── Rolling margin average ─────────────────────────────────────────
    # Extra KPI — available in fact_inventory_full, wasn't in old table
    .withColumn("avg_margin_7d", round(avg("gross_margin_pct").over(w7), 2))

    # ── KPI: Seasonality Index ─────────────────────────────────────────
    # = 7-day avg units / 90-day avg units
    .withColumn("seasonality_index",
        round(
            when(col("avg_units_90d") > 0,
                 col("avg_units_7d") / col("avg_units_90d"))
            .otherwise(lit(1.0)),
            3
        )
    )

    # ── KPI: Trend Direction ───────────────────────────────────────────
    # Compares 7-day avg vs 30-day avg to detect short-term momentum
    # Rising  → selling 10%+ faster than last month
    # Falling → selling 10%+ slower than last month
    # Stable  → within ±10% of last month
    .withColumn("trend_direction",
        when(col("avg_units_7d") > col("avg_units_30d") * 1.1, "Rising")
       .when(col("avg_units_7d") < col("avg_units_30d") * 0.9, "Falling")
       .otherwise("Stable")
    )

    # ── KPI: Stock Risk Flag ───────────────────────────────────────────
    # New KPI — cross-references demand trend with reorder_flag
    .withColumn("stock_risk_flag",
        when(
            (col("trend_direction") == "Rising") & (col("reorder_flag") == 1),
            "HIGH RISK"        # selling fast + stock below reorder level
        ).when(
            (col("trend_direction") == "Rising") & (col("stock_cover_days") < 7),
            "WATCH"            # selling fast + less than 7 days of stock left
        ).when(
            col("stockout_flag") == 1,
            "STOCKOUT"         # already zero stock
        ).otherwise("OK")
    )

    # ── Date parts for partitioning and seasonality analysis ───────────
    .withColumn("month_num",  month("snapshot_date"))
    .withColumn("year_num",   year("snapshot_date"))
    .withColumn("year_month", date_format("snapshot_date", "yyyy-MM"))

    .select(
        # Keys
        "store_id",
        "product_id",
        "snapshot_date",          
        "year_num",
        "month_num",
        "year_month",

        # Product context (available from fact_inventory_full)
        "category",
        "subcategory",
        "brand",

        # Daily actuals
        "units_sold",
        "total_revenue",          
        "gross_margin_pct",

        # Rolling demand averages
        "avg_units_7d",
        "avg_units_30d",
        "avg_units_90d",

        # Rolling revenue averages
        "avg_rev_7d",
        "avg_rev_30d",

        # Rolling margin average (new)
        "avg_margin_7d",

        # Demand KPIs
        "seasonality_index",
        "trend_direction",

        # Inventory context (from fact_inventory_full)
        "available_stock_qty",
        "stock_cover_days",
        "reorder_flag",
        "stockout_flag",
        "sell_through_rate",

        # Cross KPI (new — demand + stock combined)
        "stock_risk_flag",
    )
)

# ── Step 4: Write to Gold ──────────────────────────────────────────────
(
    demand.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year_month")
    .saveAsTable("mini_project_team_a3t7.gold.demand_intelligence1")
)

print("✅  gold.demand_intelligence written successfully.")

